# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MUKUL-TIWARI/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1

The research reports that AI-assisted content optimization improves search performance.

Methodology question:

How was improvement measured, what was the label definition, and was the comparison performed on an independent validation set rather than the same data used for development?

---

## Finding 2

The paper reports that pages with stronger search visibility generally perform better.

Methodology question:

Was this observation consistent across different clients, industries, and time periods, or was it measured from a limited subset of the data?

These questions are intended to improve confidence in the reported findings rather than challenge them.

In [1]:
%pip -q install duckdb huggingface_hub pandas numpy scikit-learn


In [7]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import accuracy_score

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass("HF Token: ")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL="hf://datasets/FlyRank/internship-warehouse"

TABLES={
"fact_daily":f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
}

query=f"""
SELECT
client_hash_id,
content_hash_id,
gsc_impressions,
gsc_clicks,
gsc_avg_position,
COALESCE(ga4_engaged_sessions,0) AS ga4_engaged_sessions
FROM {TABLES["fact_daily"]}
WHERE month='2026-03'
AND gsc_impressions>=100
LIMIT 300000
"""

df=con.sql(query).df()

df["label"]=(df["ga4_engaged_sessions"]==0).astype(int)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## Honest validation

The Week-5 model was evaluated again using a grouped split based on client identifiers. This reduces the chance that information from the same client appears in both training and testing data.

In [3]:
X=df[[
"gsc_impressions",
"gsc_clicks",
"gsc_avg_position"
]]

y=df["label"]

groups=df["client_hash_id"]

# Random split
X_train,X_test,y_train,y_test=train_test_split(
X,
y,
test_size=0.2,
random_state=42,
stratify=y
)

rf=RandomForestClassifier(
n_estimators=200,
random_state=42,
n_jobs=-1
)

rf.fit(X_train,y_train)

random_accuracy=accuracy_score(
y_test,
rf.predict(X_test)
)

# Group split
gss=GroupShuffleSplit(
n_splits=1,
test_size=0.2,
random_state=42
)

train_idx,test_idx=next(
gss.split(X,y,groups)
)

rf.fit(X.iloc[train_idx],y.iloc[train_idx])

group_accuracy=accuracy_score(
y.iloc[test_idx],
rf.predict(X.iloc[test_idx])
)

comparison=pd.DataFrame({
"Validation":[
"Random Split",
"Grouped Split"
],
"Accuracy":[
random_accuracy,
group_accuracy
]
})

comparison


,Validation,Accuracy
0,Random Split,0.971900
1,Grouped Split,0.967402


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## Leakage audit

The final feature set was reviewed for leakage.

Observed checks:

- No future information was used.
- No outcome variable was included as an input feature.
- Features were calculated from the same observation window.
- Client identifiers were used only for grouped validation and not as predictive features.

No obvious leakage was observed in the final model.

In [4]:
pd.DataFrame({
"Feature":X.columns
})


,Feature
0,gsc_impressions
1,gsc_clicks
2,gsc_avg_position


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Claim rewrite

Original claim:

The model predicts which pages require review.

Rewritten claim:

On the March 2026 dataset, the model showed measured performance under the selected validation design and may support editorial review decisions. The results should be interpreted as decision-support rather than proof of future performance.

In [5]:
importance=pd.DataFrame({
"Feature":X.columns,
"Importance":rf.feature_importances_
}).sort_values("Importance",ascending=False)

importance


,Feature,Importance
2,gsc_avg_position,0.574788
0,gsc_impressions,0.317188
1,gsc_clicks,0.108024


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.